In [5]:
# ==============================================================================
# CELL 1: DJANGO BOOTSTRAP — chạy cell này TRƯỚC TIÊN, mọi cell khác phụ thuộc
# ==============================================================================
import sys
import os

os.chdir("/home/duoc/app")        # chuyển về thư mục project
sys.path.insert(0, "/home/duoc/app")  # đảm bảo Python tìm thấy config/

os.environ.setdefault("DJANGO_SETTINGS_MODULE", "config.settings.local")
# Khai báo settings module trước khi import bất cứ thứ gì của Django
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "config.settings.local")

import django
django.setup()  # Bắt buộc — khởi tạo App Registry trước khi dùng ORM

print("Django setup OK —", django.get_version())
print("Settings module  :", os.environ["DJANGO_SETTINGS_MODULE"])

import nest_asyncio
nest_asyncio.apply()

import sys
import os

sys.path.append("/app")

from langgraph.graph import StateGraph, END, START
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import RetryPolicy

from agent_os.system.infra.telemetry import setup_observability
from agent_os.system.shields.shield_faults import PipelineError
from agent_os.system.infra.persistence import factory
from agent_os.system.bus.main_bus import MainBus
from agent_os.system.bus.registry import BusRegistry
from agent_os.system.shields.shield_node import Node

# =========================================================
# MAPPING: CORE AGENT FLOW (Theo sơ đồ Mermaid)
# =========================================================

from agent_os.nodes_library.node_aggregator.adapter_aggregator import node_aggregator
from agent_os.nodes_library.node_evaluator.adapter_evaluator import node_evaluator
from agent_os.nodes_library.node_final_response.adapter_final_response import node_final_response
from agent_os.nodes_library.node_human_review.adapter_human_review import node_human_review
from agent_os.nodes_library.node_input_guard.adapter_input_guard import node_input_guard
from agent_os.nodes_library.node_knowledge.adapter_knowledge import node_knowledge
from agent_os.nodes_library.node_marketing.adapter_node_marketing import node_marketing
from agent_os.nodes_library.node_output_guard.adapter_output_guard import node_output_guard
from agent_os.nodes_library.node_shared_state.adapter_shared_state import node_shared_state
from agent_os.nodes_library.node_supervisor.adapter_supervisor import node_supervisor
from agent_os.nodes_library.node_lightweight_chat.adapter_lightweight_chat import node_lightweight_chat



Django setup OK — 6.0.5
Settings module  : config.settings.local


In [6]:
setup_observability()

# =========================================================
# RETRY POLICY
# =========================================================
_max = RetryPolicy(
    max_attempts=2,
    backoff_factor=1.5,
    retry_on=PipelineError,
)

# =========================================================
# BOARD
# =========================================================
board = StateGraph(MainBus)

# =========================================================
# NODES
# =========================================================
Node(BusRegistry.IG,  node_input_guard).mount(board)
Node(BusRegistry.LWC, node_lightweight_chat).mount(board)
Node(BusRegistry.AG,  node_aggregator).mount(board)
Node(BusRegistry.HR,  node_human_review).mount(board)
Node(BusRegistry.OG,  node_output_guard).mount(board)
Node(BusRegistry.FR,  node_final_response).mount(board)
Node(BusRegistry.MT,  node_marketing).retry(_max).mount(board)
Node(BusRegistry.KL,  node_knowledge).retry(_max).mount(board)
Node(BusRegistry.EV,  node_evaluator).retry(_max).mount(board)
Node(BusRegistry.SV,  node_supervisor).retry(_max).mount(board)

# =========================================================
# EDGES
# =========================================================
board.add_edge(START, BusRegistry.IG)
board.add_edge(BusRegistry.IG, BusRegistry.SV)

# Supervisor → Workers
board.add_conditional_edges(
    BusRegistry.SV,
    lambda state: state.route("supervisor", default="end"),
    {
        "knowledge": BusRegistry.KL,
        "marketing": BusRegistry.MT,
        "smalltalk": BusRegistry.LWC,
        "end":       END,
    },
)

# Smalltalk shortcut
board.add_edge(BusRegistry.LWC, BusRegistry.FR)

# Workers → Evaluator
board.add_edge(BusRegistry.MT, BusRegistry.EV)
board.add_edge(BusRegistry.KL, BusRegistry.EV)

# Evaluator routing
board.add_conditional_edges(
    BusRegistry.EV,
    lambda state: state.route("evaluator", default="end"),
    {
        "retry": BusRegistry.SV,
        "pass":  BusRegistry.AG,
        "end":   END,
    },
)

# Output pipeline
board.add_edge(BusRegistry.AG, BusRegistry.OG)
board.add_edge(BusRegistry.OG, BusRegistry.HR)

# Human review routing
board.add_conditional_edges(
    BusRegistry.HR,
    lambda state: state.route("human_review", default="end"),
    {
        "approved": BusRegistry.FR,
        "rejected": BusRegistry.SV,
        "end":      END,
    },
)

# Final
board.add_edge(BusRegistry.FR, END)

# =========================================================
# COMPILE
# =========================================================
async def get_industrial_mainboard():
    async with factory.get_checkpointer() as db_checkpointer:
        return board.compile(
            checkpointer=db_checkpointer,
            interrupt_after=[BusRegistry.HR],
        )

main_v9 = board.compile(
    checkpointer=MemorySaver(),
    interrupt_after=[BusRegistry.HR], # Sử dụng giá trị "human_review"
)

# =========================================================
# DEBUG
# =========================================================
if __name__ == "__main__":
    print("\n===== MAINBOARD STRUCTURE =====")
    graph = main_v9.get_graph()
    for node in graph.nodes:
        print(f"Node: {node}")
    print("\n✅ MAINBOARD READY")

✅ Observability Layer: LangSmith Tracing Enabled.

===== MAINBOARD STRUCTURE =====
Node: __start__
Node: input_guard
Node: lightweight_chat
Node: aggregator
Node: human_review
Node: output_guard
Node: final_response
Node: marketing
Node: knowledge
Node: evaluator
Node: supervisor
Node: __end__

✅ MAINBOARD READY


In [8]:
from agent_os.system.main_app import invoke
import asyncio
import json
import uuid

# 1. Gọi hàm
random_thread_id = f"alex_test_{uuid.uuid4().hex[:8]}"
result = await asyncio.create_task(
    invoke(main_v9, "Giờ làm việc công ty", thread_id=random_thread_id)
)

# 2. Xử lý in ra JSON để dễ copy
if result and result.get("ok"):
    json_output = json.dumps(result["result"], indent=4, ensure_ascii=False)
    
    print("--- SAO CHÉP ĐOẠN DƯỚI ĐÂY ---")
    print(json_output)
    print("--- HẾT ---")
    
    from IPython.display import display, Markdown, JSON
    display(Markdown("### ✅ Kết quả thực thi"))
    display(JSON(result["result"]))
elif result:
    print(f"❌ Lỗi thực thi: {result.get('error', 'Unknown')}")
else:
    print("❌ Lỗi: Hàm invoke trả về None!")

INFO 2026-05-27 11:55:20,738 adapter_knowledge 26057 123888923869312 ⚡ [GRAPH NODE] Tiến vào phân tầng truy xuất tri thức: [node_knowledge]
INFO 2026-05-27 11:55:20,738 retrieval_service 26057 123888923869312 [Retrieval] Khởi chạy truy xuất chuẩn LlamaIndex: query=Tra cứu thông tin về các công cụ hỗ trợ công việc


--- SAO CHÉP ĐOẠN DƯỚI ĐÂY ---
{
    "user_input": "Giờ làm việc công ty",
    "input_guard": {
        "execution_id": "77a503cc-ad52-4bac-b97a-1a3b0b02a93a",
        "trace_id": "44cf8ff7-2407-42ce-8610-ddb011a5f953",
        "node_id": "input_guard",
        "payload": {
            "status": "SUCCESS",
            "text": "Giờ làm việc công ty",
            "state": {
                "is_safe": true,
                "sanitized_text": "Giờ làm việc công ty",
                "risk_category": "NONE",
                "user_input": "Giờ làm việc công ty",
                "history": [
                    {
                        "role": "user",
                        "content": "Giờ làm việc công ty"
                    },
                    {
                        "role": "assistant",
                        "content": "acknowledged"
                    }
                ],
                "user_profile": {
                    "user_id": "mock_user_001",
                    "person

### ✅ Kết quả thực thi

<IPython.core.display.JSON object>

In [4]:
# from IPython.display import HTML

# def draw_local_mermaid(graph):
#     mermaid_code = graph.draw_mermaid()
#     # Code này load thẳng thư viện Mermaid từ CDN và render tại chỗ
#     html = f"""
#     <pre class="mermaid">
#     {mermaid_code}
#     </pre>
#     <script type="module">
#         import mermaid from 'https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.esm.min.mjs';
#         mermaid.initialize({{ startOnLoad: true }});
#     </script>
#     """
#     display(HTML(html))

# draw_local_mermaid(main_v3.get_graph())

# code = main_v3.get_graph().draw_mermaid()
# print(code)

from IPython.display import display, Markdown

mermaid_code = main_v9.get_graph().draw_mermaid()

display(Markdown(f"""
```mermaid
{mermaid_code}

"""))


```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	input_guard(input_guard)
	lightweight_chat(lightweight_chat)
	aggregator(aggregator)
	human_review(human_review<hr/><small><em>__interrupt = after</em></small>)
	output_guard(output_guard)
	final_response(final_response)
	marketing(marketing)
	knowledge(knowledge)
	evaluator(evaluator)
	supervisor(supervisor)
	__end__([<p>__end__</p>]):::last
	__start__ --> input_guard;
	aggregator --> output_guard;
	evaluator -. &nbsp;end&nbsp; .-> __end__;
	evaluator -. &nbsp;pass&nbsp; .-> aggregator;
	evaluator -. &nbsp;retry&nbsp; .-> supervisor;
	human_review -. &nbsp;end&nbsp; .-> __end__;
	human_review -. &nbsp;approved&nbsp; .-> final_response;
	human_review -. &nbsp;rejected&nbsp; .-> supervisor;
	input_guard --> supervisor;
	knowledge --> evaluator;
	lightweight_chat --> final_response;
	marketing --> evaluator;
	output_guard --> human_review;
	supervisor -. &nbsp;end&nbsp; .-> __end__;
	supervisor -.-> knowledge;
	supervisor -. &nbsp;smalltalk&nbsp; .-> lightweight_chat;
	supervisor -.-> marketing;
	final_response --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc




In [5]:
code = main_v9.get_graph().draw_mermaid()
print(code)

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	input_guard(input_guard)
	lightweight_chat(lightweight_chat)
	aggregator(aggregator)
	human_review(human_review<hr/><small><em>__interrupt = after</em></small>)
	output_guard(output_guard)
	final_response(final_response)
	marketing(marketing)
	knowledge(knowledge)
	evaluator(evaluator)
	supervisor(supervisor)
	__end__([<p>__end__</p>]):::last
	__start__ --> input_guard;
	aggregator --> output_guard;
	evaluator -. &nbsp;end&nbsp; .-> __end__;
	evaluator -. &nbsp;pass&nbsp; .-> aggregator;
	evaluator -. &nbsp;retry&nbsp; .-> supervisor;
	human_review -. &nbsp;end&nbsp; .-> __end__;
	human_review -. &nbsp;approved&nbsp; .-> final_response;
	human_review -. &nbsp;rejected&nbsp; .-> supervisor;
	input_guard --> supervisor;
	knowledge --> evaluator;
	lightweight_chat --> final_response;
	marketing --> evaluator;
	output_guard --> human_review;
	supervisor -. &nbsp;end&nbsp; .-> __end__;
	supervis

In [4]:
# Xem httpx version và cách litellm gọi ollama
import httpx
print(httpx.__version__)

import litellm
print(litellm.__version__)

0.28.1


AttributeError: module 'litellm' has no attribute '__version__'

In [5]:
# Test gọi Ollama trực tiếp — bypass litellm hoàn toàn
import httpx
import asyncio

async def test_ollama_direct():
    async with httpx.AsyncClient(timeout=30) as client:
        resp = await client.post(
            "http://localhost:11434/api/chat",
            json={
                "model": "qwen2.5:7b",  # thay bằng model bạn đang dùng
                "messages": [{"role": "user", "content": "hello"}],
                "stream": False
            }
        )
        print(resp.json())

await asyncio.create_task(test_ollama_direct())

AsyncLibraryNotFoundError: unknown async library, or not in async context